# Inventaire et premiers contrôles

Ce notebook vérifie les fichiers présents et affiche `info()` et `describe()` pour les données disponibles. Le jeu Fraud est signalé sans interrompre l'analyse s'il manque. Le CSV Lending Club est limité à un échantillon de 20 000 lignes pour éviter de charger plusieurs gigaoctets en mémoire.

Lancer Jupyter depuis la racine du dépôt. Les jeux ne sont jamais concaténés entre eux.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "data" / "raw").exists():
    ROOT_DIR = ROOT_DIR.parent

RAW_DIR = ROOT_DIR / "data" / "raw"
print(f"Données brutes : {RAW_DIR}")

## Fonction de contrôle

Pour les CSV très volumineux, `nrows` permet de limiter la lecture. Chaque résultat décrit uniquement le fichier et, le cas échéant, l'échantillon indiqué.

In [ ]:
def inspect_dataset(name, relative_path, *, separator=",", nrows=None, excel_header=0):
    """Charger un fichier disponible et afficher ses premiers contrôles."""
    path = RAW_DIR / relative_path
    if not path.is_file():
        print(f"\n{name} : fichier absent ({path})")
        return None

    if path.suffix.lower() == ".xls":
        frame = pd.read_excel(path, header=excel_header, engine="calamine")
    else:
        frame = pd.read_csv(path, sep=separator, nrows=nrows, low_memory=False)

    print(f"\n{name} — {path.name} — forme : {frame.shape}")
    frame.info(memory_usage="deep")
    print("\nStatistiques descriptives (aperçu des 12 premières variables) :")
    display(frame.describe(include="all").T.head(12))
    return frame

## Home Credit — jeu prioritaire

On commence par la table d'entraînement principale. Les agrégations multi-tables seront réalisées dans une étape dédiée, après l'examen des clés et de la qualité des données.

In [ ]:
home_credit_train = inspect_dataset(
    "Home Credit (train)",
    "home_credit/application_train.csv",
)

if home_credit_train is not None:
    print("\nRépartition de TARGET :")
    display(home_credit_train["TARGET"].value_counts(dropna=False).rename("effectif"))
    display(home_credit_train["TARGET"].value_counts(normalize=True).rename("proportion"))

## Jeux complémentaires disponibles

Les fichiers Lending Club ne sont lus que sur les 20 000 premières lignes de chaque CSV dans ce premier contrôle. Une analyse représentative complète nécessitera une lecture par blocs ou une sélection raisonnée des colonnes.

In [ ]:
uci_credit_card = inspect_dataset(
    "UCI Credit Card Clients",
    "uci_credit_card/default of credit card clients.xls",
    excel_header=1,
)

bank_marketing = inspect_dataset(
    "Bank Marketing (full)",
    "bank_marketing/bank-full.csv",
    separator=";",
)

bank_additional = inspect_dataset(
    "Bank Marketing Additional (full)",
    "bank_marketing/bank-additional-full.csv",
    separator=";",
)

lending_accepted_sample = inspect_dataset(
    "Lending Club accepted (échantillon)",
    "lending_club/accepted_2007_to_2018Q4.csv",
    nrows=20_000,
)

lending_rejected_sample = inspect_dataset(
    "Lending Club rejected (échantillon)",
    "lending_club/rejected_2007_to_2018Q4.csv",
    nrows=20_000,
)

fraud_sample = inspect_dataset(
    "Credit Card Fraud",
    "fraud/creditcard.csv",
)

target_columns = [
    ("UCI Credit Card Clients", uci_credit_card, "default payment next month"),
    ("Bank Marketing", bank_marketing, "y"),
    ("Bank Marketing Additional", bank_additional, "y"),
    ("Credit Card Fraud", fraud_sample, "Class"),
]

for name, frame, target in target_columns:
    if frame is not None:
        print(f"\n{name} — répartition de {target} :")
        display(frame[target].value_counts(dropna=False).rename("effectif"))
        display(frame[target].value_counts(normalize=True, dropna=False).rename("proportion"))